# Week 7: Data Pipelines & Automation
## Weather Data ETL Pipeline
### Python + Pandas + OpenWeather API

## 1. Extract

In [29]:
# Import requests for making API calls
import requests

# Import pandas for organizing and transforming the weather data
import pandas as pd

In [30]:
# Install python-dotenv so we can safely load the API key
# from a local environment file.
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [31]:
# Create a local .env file to store the OpenWeather API key
with open(".env", "w") as file:
    file.write("OPENWEATHER_API_KEY=###")

In [32]:
# Save your OpenWeather API key locally
with open(".env", "w") as file:
    file.write("OPENWEATHER_API_KEY=###")

In [33]:
# Prevent the .env file containing the API key from being uploaded to GitHub
with open(".gitignore", "w") as file:
    file.write(".env\n")

In [34]:
# Import the function used to load variables from the .env file
from dotenv import load_dotenv

# Import os so we can access the stored API key
import os

# Load the variables from the .env file
load_dotenv()

# Retrieve the OpenWeather API key
API_KEY = os.getenv("OPENWEATHER_API_KEY")

# Confirm that the key was loaded without displaying the actual key
if API_KEY:
    print("API key loaded successfully.")
else:
    print("API key was not found.")

API key loaded successfully.


In [35]:
# Set the city we want to retrieve weather data for
city = "Lagos"

# Build the OpenWeather API endpoint
url = "https://api.openweathermap.org/data/2.5/weather"

# Define the parameters for our API request
params = {
    "q": city,
    "appid": API_KEY,
    "units": "metric"
}

# Send the request to OpenWeather
response = requests.get(url, params=params)

# Display the HTTP status code
print("Status code:", response.status_code)

# Display the returned weather data
print(response.json())

Status code: 200
{'coord': {'lon': 3.75, 'lat': 6.5833}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 27.39, 'feels_like': 29.95, 'temp_min': 27.39, 'temp_max': 27.39, 'pressure': 1014, 'humidity': 74, 'sea_level': 1014, 'grnd_level': 1014}, 'visibility': 10000, 'wind': {'speed': 4.05, 'deg': 234, 'gust': 6.15}, 'clouds': {'all': 93}, 'dt': 1786966910, 'sys': {'country': 'NG', 'sunrise': 1786945170, 'sunset': 1786989542}, 'timezone': 3600, 'id': 2332453, 'name': 'Lagos', 'cod': 200}


In [36]:
# Function to extract weather data for a given city
def get_weather(city):
    # OpenWeather API endpoint for current weather
    url = "https://api.openweathermap.org/data/2.5/weather"

    # Parameters sent with the API request
    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    # Send the request to the API
    response = requests.get(url, params=params)

    # Check whether the request was successful
    if response.status_code == 200:
        # Return the raw JSON response
        return response.json()
    else:
        # Return an error message if the request fails
        print(f"Error retrieving data for {city}: {response.status_code}")
        return None

In [37]:
# List of cities we want to collect weather data for
cities = ["Lagos", "London", "New York"]

# Extract the raw weather data for each city
weather_data = []

for city in cities:
    data = get_weather(city)

    # Add the successful API response to our collection
    if data is not None:
        weather_data.append(data)

# Confirm how many cities were successfully extracted
print(f"Successfully extracted data for {len(weather_data)} cities.")

Successfully extracted data for 3 cities.


In [38]:
# Display the raw weather response for each city
for data in weather_data:
    print(data["name"])
    print(data)
    print("-" * 80)

Lagos
{'coord': {'lon': 3.75, 'lat': 6.5833}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 27.39, 'feels_like': 29.95, 'temp_min': 27.39, 'temp_max': 27.39, 'pressure': 1014, 'humidity': 74, 'sea_level': 1014, 'grnd_level': 1014}, 'visibility': 10000, 'wind': {'speed': 4.05, 'deg': 234, 'gust': 6.15}, 'clouds': {'all': 93}, 'dt': 1786966910, 'sys': {'country': 'NG', 'sunrise': 1786945170, 'sunset': 1786989542}, 'timezone': 3600, 'id': 2332453, 'name': 'Lagos', 'cod': 200}
--------------------------------------------------------------------------------
London
{'coord': {'lon': -0.1257, 'lat': 51.5085}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 25.24, 'feels_like': 25.15, 'temp_min': 23.86, 'temp_max': 27.23, 'pressure': 1016, 'humidity': 51, 'sea_level': 1016, 'grnd_level': 1012}, 'visibility': 10000, 'wind': {'speed':

## 2. Transform

In [39]:
# Create an empty list to store the transformed records
transformed_data = []

# Loop through each city's raw API response
for data in weather_data:
    
    # Extract the required fields from the nested API response
    record = {
        "City": data["name"],
        "Temperature_C": data["main"]["temp"],
        "Humidity_Percent": data["main"]["humidity"],
        "Weather_Condition": data["weather"][0]["description"],
        "Wind_Speed_mps": data["wind"]["speed"],
        "Timestamp": data["dt"]
    }
    
    # Add the cleaned record to our list
    transformed_data.append(record)

# Convert the list of records into a Pandas DataFrame
weather_df = pd.DataFrame(transformed_data)

# Display the transformed dataset
weather_df

,City,Temperature_C,Humidity_Percent,Weather_Condition,Wind_Speed_mps,Timestamp
0,Lagos,27.39,74,overcast clouds,4.05,1786966910
1,London,25.24,51,broken clouds,2.68,1786966589
2,New York,21.81,96,overcast clouds,4.47,1786966918


In [40]:
# Convert the Unix timestamp into a readable date and time
weather_df["Date_Time"] = pd.to_datetime(
    weather_df["Timestamp"],
    unit="s"
)

# Remove the original Unix timestamp column
weather_df = weather_df.drop(columns=["Timestamp"])

# Display the updated dataset
weather_df

,City,Temperature_C,Humidity_Percent,Weather_Condition,Wind_Speed_mps,Date_Time
0,Lagos,27.39,74,overcast clouds,4.05,2026-08-17 11:41:50
1,London,25.24,51,broken clouds,2.68,2026-08-17 11:36:29
2,New York,21.81,96,overcast clouds,4.47,2026-08-17 11:41:58


In [41]:
# Convert the weather measurements to numeric data types
weather_df["Temperature_C"] = pd.to_numeric(weather_df["Temperature_C"])
weather_df["Humidity_Percent"] = pd.to_numeric(weather_df["Humidity_Percent"])
weather_df["Wind_Speed_mps"] = pd.to_numeric(weather_df["Wind_Speed_mps"])

# Check the data types of all columns
weather_df.dtypes

City                         object
Temperature_C               float64
Humidity_Percent              int64
Weather_Condition            object
Wind_Speed_mps              float64
Date_Time            datetime64[ns]
dtype: object

In [42]:
# Check each column for missing values
print("Missing values:")
print(weather_df.isnull().sum())

# Check for duplicate rows
print("\nDuplicate rows:")
print(weather_df.duplicated().sum())

Missing values:
City                 0
Temperature_C        0
Humidity_Percent     0
Weather_Condition    0
Wind_Speed_mps       0
Date_Time            0
dtype: int64

Duplicate rows:
0


In [43]:
# Save the transformed weather data as a CSV file
weather_df.to_csv("processed_weather_data.csv", index=False)

# Confirm that the file was saved successfully
print("Processed weather data saved successfully.")

Processed weather data saved successfully.


In [44]:
# Compare the temperatures across the three cities
temperature_comparison = weather_df[["City", "Temperature_C"]]

# Display the temperature comparison
temperature_comparison

,City,Temperature_C
0,Lagos,27.39
1,London,25.24
2,New York,21.81


In [45]:
# Identify the city with the highest temperature
hottest_city = weather_df.loc[
    weather_df["Temperature_C"].idxmax()
]

# Identify the city with the highest humidity
most_humid_city = weather_df.loc[
    weather_df["Humidity_Percent"].idxmax()
]

# Display the results
print(
    f"Hottest city: {hottest_city['City']} "
    f"({hottest_city['Temperature_C']:.2f}°C)"
)

print(
    f"Most humid city: {most_humid_city['City']} "
    f"({most_humid_city['Humidity_Percent']}%)"
)

Hottest city: Lagos (27.39°C)
Most humid city: New York (96%)


In [46]:
# Count how many cities currently have each weather condition
weather_condition_summary = (
    weather_df["Weather_Condition"]
    .value_counts()
    .reset_index()
)

# Rename the columns for clarity
weather_condition_summary.columns = [
    "Weather_Condition",
    "City_Count"
]

# Display the summary
weather_condition_summary

,Weather_Condition,City_Count
0,overcast clouds,2
1,broken clouds,1


In [47]:
# Add the country code from the raw API response to each city's record
for i, data in enumerate(weather_data):
    weather_df.loc[i, "Country"] = data["sys"]["country"]

# Display the updated dataset
weather_df

,City,Temperature_C,Humidity_Percent,Weather_Condition,Wind_Speed_mps,Date_Time,Country
0,Lagos,27.39,74,overcast clouds,4.05,2026-08-17 11:41:50,NG
1,London,25.24,51,broken clouds,2.68,2026-08-17 11:36:29,GB
2,New York,21.81,96,overcast clouds,4.47,2026-08-17 11:41:58,US


## 3. Load

In [48]:
# Save the final transformed dataset with the country information included
weather_df.to_csv("processed_weather_data.csv", index=False)

# Confirm that the final dataset was saved
print("Final processed weather dataset saved successfully.")

Final processed weather dataset saved successfully.


In [49]:
# Create the Python ETL pipeline script
script = '''
# ============================================
# WEATHER ETL PIPELINE
# Extract, Transform, Load
# ============================================

import os
import requests
import pandas as pd
from dotenv import load_dotenv


# ============================================
# 1. EXTRACT
# ============================================

load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")

URL = "https://api.openweathermap.org/data/2.5/weather"

cities = ["Lagos", "London", "New York"]


def get_weather(city):
    # Send a request to the OpenWeather API
    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    response = requests.get(URL, params=params)

    # Return the raw response if the request succeeds
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error retrieving data for {city}: {response.status_code}")
        return None


# Extract data for all cities
weather_data = []

for city in cities:
    data = get_weather(city)

    if data is not None:
        weather_data.append(data)

print(f"Successfully extracted data for {len(weather_data)} cities.")


# ============================================
# 2. TRANSFORM
# ============================================

transformed_data = []

for data in weather_data:
    record = {
        "City": data["name"],
        "Country": data["sys"]["country"],
        "Temperature_C": data["main"]["temp"],
        "Humidity_Percent": data["main"]["humidity"],
        "Weather_Condition": data["weather"][0]["description"],
        "Wind_Speed_mps": data["wind"]["speed"],
        "Date_Time": pd.to_datetime(data["dt"], unit="s")
    }

    transformed_data.append(record)


# Convert the extracted records into a DataFrame
weather_df = pd.DataFrame(transformed_data)

# Ensure numerical columns have the correct data types
weather_df["Temperature_C"] = pd.to_numeric(weather_df["Temperature_C"])
weather_df["Humidity_Percent"] = pd.to_numeric(weather_df["Humidity_Percent"])
weather_df["Wind_Speed_mps"] = pd.to_numeric(weather_df["Wind_Speed_mps"])

# Check for missing values
print("\\nMissing values:")
print(weather_df.isnull().sum())

# Check for duplicate rows
print("\\nDuplicate rows:")
print(weather_df.duplicated().sum())


# ============================================
# 3. LOAD
# ============================================

# Save the transformed data as a CSV file
weather_df.to_csv("processed_weather_data.csv", index=False)

print("\\nProcessed weather data saved successfully.")

# Display the final dataset
print("\\nFinal Dataset:")
print(weather_df)
'''

# Write the script to a .py file
with open("weather_etl_pipeline.py", "w", encoding="utf-8") as file:
    file.write(script)

print("ETL pipeline script created successfully.")

ETL pipeline script created successfully.


In [50]:
# Run the complete ETL pipeline script
%run weather_etl_pipeline.py

Successfully extracted data for 3 cities.

Missing values:
City                 0
Country              0
Temperature_C        0
Humidity_Percent     0
Weather_Condition    0
Wind_Speed_mps       0
Date_Time            0
dtype: int64

Duplicate rows:
0

Processed weather data saved successfully.

Final Dataset:
       City Country  Temperature_C  Humidity_Percent Weather_Condition  \
0     Lagos      NG          28.57                69     broken clouds   
1    London      GB          25.52                50     broken clouds   
2  New York      US          21.80                95        light rain   

   Wind_Speed_mps           Date_Time  
0            3.83 2026-08-17 12:05:10  
1            1.34 2026-08-17 12:08:31  
2            5.66 2026-08-17 12:15:08  


In [52]:
# Create the final README.md file

readme = """
# Weather Data ETL Pipeline

## Project Overview

This project demonstrates a basic Extract, Transform, Load (ETL) pipeline using real-time weather data from the OpenWeather API.

Weather data was collected for Lagos, London, and New York using Python. The raw API responses were transformed into a clean and structured Pandas DataFrame, validated, and stored as a CSV file for analysis.

## Objective

The objective of this project is to demonstrate how Python can be used to:

- Extract data from an external API
- Transform raw JSON data into a structured format
- Clean and validate the dataset
- Store processed data for future analysis
- Perform basic analysis on the resulting dataset

## Data Source

The weather data was obtained from the OpenWeather API.

The API provided information including:

- City
- Country
- Temperature
- Humidity
- Weather condition
- Wind speed
- Date and time

## ETL Process

### Extract

The OpenWeather API was accessed using Python and the Requests library.

Weather data was collected for three cities:

- Lagos
- London
- New York

The API returned the weather information as nested JSON responses.

### Transform

The raw API responses were transformed using Pandas.

The transformation process included:

- Extracting required fields from nested JSON responses
- Creating a structured Pandas DataFrame
- Standardizing column names
- Converting Unix timestamps into readable datetime values
- Converting numerical fields to appropriate data types
- Checking for missing values
- Checking for duplicate records

The final dataset contained zero missing values and zero duplicate records.

### Load

The transformed dataset was saved as:

`processed_weather_data.csv`

The processed CSV file can be used for further analysis and visualization.

## Basic Analysis

The processed dataset was analyzed to compare temperature, humidity, weather conditions, and wind speed across the three selected cities.

## Key Findings

- Lagos recorded the highest temperature at **28.57°C**.
- New York recorded the highest humidity at **95%**.
- New York recorded the highest wind speed at **5.66 m/s**.
- Lagos and London reported **broken clouds**.
- New York reported **light rain**.
- Weather data was successfully extracted for all three cities.
- The final dataset contained **zero missing values** and **zero duplicate records**.

These findings represent a real-time weather snapshot at the time of data extraction and should not be interpreted as long-term climate trends.

## Tools Used

- Python
- Pandas
- Requests
- python-dotenv
- Jupyter Notebook
- OpenWeather API

## Project Structure

```text
Weather-ETL-Pipeline/
│
├── Weather_ETL_Analysis.ipynb
├── weather_etl_pipeline.py
├── processed_weather_data.csv
├── README.md
└── .gitignore

The .env file containing the private OpenWeather API key is intentionally excluded from the repository using .gitignore.

What I Learned

This project provided practical experience in building an ETL pipeline using Python.

I learned how to:

-Connect Python to an external API
-Retrieve and work with JSON data
-Extract specific fields from nested API responses
-Use Pandas to structure and transform data
-Convert Unix timestamps into readable datetime values
-Validate datasets by checking for missing and duplicate records
-Store transformed data as a CSV file
-Separate API credentials from source code
-Organize an ETL workflow into Extract, Transform, and Load stages
-The project also reinforced the importance of data validation and secure handling of API credentials when working with external data sources.

Conclusion
-The project successfully demonstrates a basic automated ETL workflow.
-Weather data was extracted from the OpenWeather API for Lagos, London, and New York. The raw JSON responses were transformed into a clean Pandas DataFrame, validated for data quality, and loaded into a CSV file.
-The resulting dataset was then used to perform basic comparisons of temperature, humidity, weather conditions, and wind speed across the selected cities.
Overall, the project demonstrates how an ETL pipeline can convert raw API data into a structured dataset that is ready for analysis.

Future Improvements
-The pipeline could be expanded by:
-Collecting weather data from more cities
-Scheduling the pipeline to run automatically
-Storing historical API results instead of only the latest snapshot
-Adding more advanced data visualizations
-Storing the data in a SQLite database
-Building a dashboard to monitor weather trends over time
"""

with open("README.md", "w", encoding="utf-8") as file:
    file.write(readme)

print("README.md created successfully.")

README.md created successfully.
